In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import psycopg2
import boto3
import numpy as np

In [2]:
storage_options = {
    "key": "admin",
    "secret": "admin123",
    "client_kwargs": {"endpoint_url": "http://minio:9000"}
}
RAW = "s3://raw"
PROCESSED = "s3://processed"

In [3]:
wells = pd.read_parquet(
    f"{RAW}/wells/",           
    storage_options=storage_options
)
production = pd.read_parquet(
    f"{RAW}/production/",            
    storage_options=storage_options,
)
telemetry = pd.read_parquet(
    f"{RAW}/telemetry/",            
    storage_options=storage_options,
)

/opt/conda/lib/python3.11/site-packages/fsspec/registry.py:271: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [4]:
production["date"] = pd.to_datetime(production["date"]).dt.date
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"])
telemetry["date"] = telemetry["timestamp"].dt.date

Обработка NULL

In [5]:
prod_num_cols = ["oil_ton", "gas_m3", "water_m3", "energy_kwh", "downtime_hours"]
for col in prod_num_cols:
    if col in production.columns:
        production[col] = production[col].fillna(0)
for col in ["temperature", "pressure"]:
    if col in production.columns:
        production[col] = production.groupby("well_id")[col].transform(
            lambda s: s.fillna(s.mean())
        )

In [6]:
sensor_cols = ["pump_speed_rpm", "pump_current", "pressure_in", "pressure_out",
               "temperature", "vibration", "oil_flow_rate"]
for col in sensor_cols:
    if col in telemetry.columns:
        telemetry[col] = telemetry.groupby("well_id")[col].transform(
            lambda s: s.fillna(s.mean())
        )

In [7]:
wells = wells.fillna({"region": "unknown", "operator": "unknown", "status": "active"})

Фильтр выбросов

In [8]:
def remove_outliers_iqr(df: pd.DataFrame, cols: list, k: float = 3.0) -> pd.DataFrame:
    df = df.copy()
    mask = pd.Series(True, index=df.index)
    
    for col in cols:
        if col not in df.columns:
            continue
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - k * iqr, q3 + k * iqr
        col_mask = df[col].between(low, high)
        removed = (~col_mask).sum()
        print(f"  {col}: удалено {removed} ({removed/len(df)*100:.2f}%)")
        mask &= col_mask
    
    return df[mask].reset_index(drop=True)

In [9]:
# сначала бизнес-правила (физика)
before = len(telemetry)
telemetry = telemetry[telemetry["pressure_in"] >= 0]
telemetry = telemetry[telemetry["pressure_out"] >= 0]
telemetry = telemetry[telemetry["pump_speed_rpm"] >= 0]
telemetry = telemetry[telemetry["vibration"] >= 0]
telemetry = telemetry[telemetry["temperature"].between(-50, 200)]
print(f"telemetry: после физ. правил удалено {before - len(telemetry)} строк")

telemetry: после физ. правил удалено 0 строк


In [10]:
telemetry = remove_outliers_iqr(telemetry, sensor_cols, k=3.0)

  pump_speed_rpm: удалено 0 (0.00%)
  pump_current: удалено 0 (0.00%)
  pressure_in: удалено 0 (0.00%)
  pressure_out: удалено 0 (0.00%)
  temperature: удалено 0 (0.00%)
  vibration: удалено 0 (0.00%)
  oil_flow_rate: удалено 0 (0.00%)


In [11]:
prod_iqr_cols = ["oil_ton", "gas_m3", "water_m3", "energy_kwh"]
production = remove_outliers_iqr(production, prod_iqr_cols, k=3.0)

  oil_ton: удалено 0 (0.00%)
  gas_m3: удалено 270 (20.00%)
  water_m3: удалено 0 (0.00%)
  energy_kwh: удалено 0 (0.00%)


In [12]:
production = production[production["downtime_hours"].between(0, 24)]

Агрегации

In [13]:
daily_telemetry = (
    telemetry
    .groupby(["well_id", "date"], as_index=False)
    .agg(
        avg_pump_speed=("pump_speed_rpm", "mean"),
        avg_pump_current=("pump_current", "mean"),
        avg_pressure_in=("pressure_in", "mean"),
        avg_pressure_out=("pressure_out", "mean"),
        avg_temperature_sensor=("temperature", "mean"),
        avg_vibration=("vibration", "mean"),
        max_vibration=("vibration", "max"),
        avg_oil_flow=("oil_flow_rate", "mean"),
        records_per_day=("record_id", "count"),
    )
    .round(2)
)

In [16]:
daily = production.merge(
    daily_telemetry,
    on=["well_id", "date"],
    how="left",
)

Среднее давление

In [17]:
daily["avg_pressure"] = (daily["avg_pressure_in"] + daily["avg_pressure_out"]) / 2

Средняя температура

In [18]:
daily["avg_temperature"] = daily["avg_temperature_sensor"]

Коэффициент простоя

In [22]:
daily["downtime_ratio"] = (daily["downtime_hours"] / 24).clip(0, 1)

In [24]:
daily.to_parquet(
    f"{PROCESSED}/daily_telemetry/",
    partition_cols=["date"],
    storage_options=storage_options,
    index=False,
)